<img src="https://cdn-ukwest.onetrust.com/logos/8330d093-4c49-4c87-9f9c-f0411dece48c/f4949705-ab9e-41bb-b3d9-6abc65ae7a94/8d8533f7-8206-4edc-83f0-d8eda8c92ae4/BPP_1-Line_Lockup_Positive_RGB_Web.png" width=400px/>

<h3><font color="#AA00BB">How you can use this Notebook</font></h3>
<p>This notebook was written to teach new concepts in data using Python.</p>
<p>You can read through the descriptions run the code (it should work!), or you may be taken through the code by one of our experts.</p>
<p>But one of the best habits to acquire is to re-write the code yourself.</p>
<ul><li>Experiment
<li>Break the code
<li>Build a deeper understanding of both the underlying data concepts and the code</ul>
<p>Don't worry if you make mistakes, we all do. The way to get better and make less mistakes is to write more code.</p>
<p>Enjoy!</p>
<br>

> ❓🤔 = a question for us discuss

> ⌨️ = a task for you to try

> 🔑 = an authoritative reference or guide you may find useful

> 🚀 = **optional** material to develop yourself further


<a name="contents"></a>
## Contents

<br>1. [What is Spark?](#section_1)
<br>2. [More Spark](#section_2)
<br>3. [SparkSQL](#section_3)

<a name="section_1"></a>
# 1. What is Spark?

[Return to contents](#contents)

Apache Spark is an open-source, distributed computing system designed for big data processing. It allows for fast and efficient processing of large-scale datasets by distributing tasks across a cluster of machines. Spark provides APIs in multiple languages (Python, Java, Scala, and R) and supports libraries for SQL, streaming, machine learning, and graph processing.

**Why We Use Spark:**
* Speed: Spark processes data in memory, making it faster than traditional disk-based frameworks like Hadoop.
* Ease of Use: It provides simple APIs and supports multiple languages.
* Versatility: Spark can handle batch processing, real-time data streams, and advanced analytics in a single framework.
* Scalability: It can process petabytes of data across large clusters.

**Benefits:**
* **In-Memory Processing:** Faster data processing compared to disk-based alternatives.
* **Unified Framework:** Combines multiple processing needs (batch, real-time, machine learning).
* **Rich Ecosystem:** Includes tools for advanced analytics like MLlib and GraphX.

**Limitations:**
* **High Memory Usage:** In-memory processing requires significant RAM, which can be costly.
* **Steeper Learning Curve:** Understanding distributed systems and Spark APIs may be challenging for beginners.
* **I/O Bottlenecks:** Performance may degrade if frequent data shuffling or I/O operations are required.

Spark is a powerful tool for big data processing but is best suited for environments with sufficient resources and expertise.

In [ ]:
# first let's start by installing Spark
!pip install findspark

In [ ]:
# ...and now create a spark session!
APP_NAME = "Debugging Prediction Problems"

# If there is no SparkSession, create the environment
try:
    sc and spark
except NameError as e:
    import findspark
    findspark.init()
    import pyspark
    from pyspark.sql import SparkSession

    sc = pyspark.SparkContext(appName=APP_NAME)
    spark = SparkSession.builder.appName(APP_NAME).getOrCreate()

print("PySpark initiated...")


Now you'll need to upload the file `Example.csv` to Colab using the upload button.

In [ ]:
import pandas as pd
data = [
    ['Russell Jurney', 'Relato', 'CEO'],
    ['Florian Liebert', 'Mesosphere', 'CEO'],
    ['Don Brown', 'Rocana', 'CIO'],
    ['Steve Jobs', 'Apple', 'CEO'],
    ['Donald Trump', 'The Trump Organization', 'CEO'],
    ['Russell Jurney', 'Data Syndrome', 'Principal Consultant']
    ]
pd.DataFrame(data).to_csv('employees.csv', header=False, index=False)

In [ ]:
# Loading and Collecting Data

# Load the text file using SparkContext
csv_lines = sc.textFile('employees.csv')

# Map the data to split the lines into a list
data = csv_lines.map(lambda line: line.split(","))

# Collect the dataset into local RAM
collected_data = data.collect()
print("Collected Data:", collected_data)

**Question:**
* What is the output of the above operation? Write your observations.
* Research and write below the pros and cons of running collect() in Spark:

**Pros:**
1.
2.
3.

**Cons:**
1.
2.
3.

In [ ]:
#@title Answers
# Output is a list of lists

# Pros:
# 1. Easy to debug as all data is in local memory.
# 2. Quick access to a small dataset for inspection.
# 3. Useful for educational and testing purposes.

# Cons:
# 1. Can lead to memory overload for large datasets.
# 2. Breaks the distributed nature of Spark.
# 3. Limits scalability.
# 4. High risk of out-of-memory errors.

**Question:** Is it a good idea to run collect()? Why or why not?


In [ ]:
#@title Answer

# It is generally not a good idea to run collect() on large datasets as it
# defeats the purpose of distributed computing in Spark and can lead to memory
# errors. However, for small datasets or debugging purposes, it can be useful.

In [ ]:
# Using the groupBy Operator

# Group the records by the name of the person
records = csv_lines.map(lambda line: line.split(","))
grouped_records = records.groupBy(lambda x: x[0])

# Show the first group
first_group = grouped_records.first()
print("First Group:", first_group)

# Count the groups
job_counts = grouped_records.map(
    lambda x: {
        "name": x[0],
        "job_count": len(list(x[1]))
    }
)

first_job_count = job_counts.first()
print("First Job Count:", first_job_count)
print("All Job Counts:", job_counts.collect())

**Task:**
* What is the output?
* Explain what happened. What does the output represent in plain English?


In [ ]:
#@title Answer
# The output shows that the records were grouped by the name of the person.
# Each group contains the name as the key and a list of associated records
# as the value. The job counts show the number of records for each name.

In [ ]:
# Map vs FlatMap

# Compute a relation of words by line
words_by_line = csv_lines.map(lambda line: line.split(","))
print("Words by Line:", words_by_line.collect())

# Compute a flattened relation of words
flattened_words = csv_lines.flatMap(lambda line: line.split(","))
print("Flattened Words:", flattened_words.collect())

**Question:** What is the difference between map and flatMap? Provide examples and explain in plain English.

In [ ]:
#@title Answer:
# map: Splits each line into a list, keeping the structure intact.
# flatMap: Splits each line into individual elements, flattening the result.

# Example:
# map: [['John', 'Engineer', '28'], ['Doe', 'Doctor', '34'], ['Alice', 'Artist', '29']]
# flatmap: ['John', 'Engineer', '28', 'Doe', 'Doctor', '34', 'Alice', 'Artist', '29']

In [ ]:
import kagglehub

# Download latest version
pc_path = kagglehub.dataset_download("danwinchester/open-postcode-geo")

print("Path to dataset files:", pc_path)

In [ ]:
import os
postcodes = sc.textFile(os.path.join(pc_path, 'open_postcode_geo.csv'))

# Skip the header row
header = csv_lines.first()
data = csv_lines.filter(lambda line: line != header)

postcode_coordinates = postcodes\
  .map(lambda line: line.split(","))\
  .map(lambda fields: (fields[0], (fields[-2], fields[-3])))

output = postcode_coordinates.collect()
print(output)

<a name="section_2"></a>
# 2. More Spark

[Return to contents](#contents)

Now you're going to have a look at a new example. You're going to take two files and join them together.

Steps:

1. Map separate datasets into key-value Pair RDDs
 * Map web log requests to (docid, userid)
 * Map KB Doc index to (docid, title)

2. Join by key: docid
3. Map joined data into the desired format: (userid, title)
4. Further processing: group titles by User ID

In [ ]:
# data (note this is small dataset to work with!)
weblogs = "32.54.32.111 - 93332 \"GET /KBDOC-00157.html HTTP/1.0\"\n132.54.32.212 - 93332 \"GET /theme.css  HTTP/1.0\"\n132.54.32.212 - 25254 \"GET /KBDOC-00230.html  HTTP/1.0\""
kblist = "KBDOC-00157:Parallel Programming\nKBDOC-00230:Distributed Systems\nKBDOC-00221:HCI"

In [ ]:
# a function that'lll be needed later...
import re
def getKBDOC(stringy):
    return re.search(r'KBDOC-[0-9]*',stringy).group()

In [ ]:
# get the files into RDDs
rdd1 = sc.parallelize(weblogs.split("\n"))
rdd2 = sc.parallelize(kblist.split("\n"))

In [ ]:
# Filter data so that only KBDOCs are present
rdd1_filt = rdd1.filter(lambda line: 'KBDOC' in line.split(' ')[4])

# Format the data
rdd1_form = rdd1_filt.map(lambda line: (getKBDOC(line.split(' ')[4]), line.split(' ')[2]))
rdd2_form = rdd2.map(lambda line: (line.split(':')[0],line.split(':')[1]))
print(rdd1_form.collect())
print(rdd2_form.collect())

In [ ]:
# Join the data
result = rdd1_form.join(rdd2_form)
print(result.collect())

In [ ]:
# Format the data again
result_form = result.map(lambda lines: (lines[1][0], lines[1][1]))
print(result_form.collect())

In [ ]:
result_form.groupByKey().mapValues(list).collect()

<a name="section_3"></a>
# 3. SparkSQL

[Return to contents](#contents)

### **What is SparkSQL?**

SparkSQL is a module in Apache Spark that allows users to query structured and semi-structured data using SQL. It integrates seamlessly with Spark’s distributed computing engine and supports querying data from various sources like Hive tables, Parquet files, JSON, and more.

---

### **Why Use SparkSQL Over Spark Core?**

1. **Ease of Use**:
   - For users familiar with SQL, SparkSQL provides a simple interface to query data without needing to learn the full Spark API.
   - It abstracts the complexities of distributed processing, making it accessible to a wider audience.

2. **Unified Query Language**:
   - With SparkSQL, you can use SQL syntax for querying, while still benefiting from Spark’s distributed processing capabilities.
   - It integrates SQL queries with Spark DataFrame and Dataset APIs, providing flexibility for developers.

3. **Optimizations**:
   - SparkSQL uses the **Catalyst Optimizer**, which generates efficient execution plans for SQL queries, improving performance.
   - It can handle complex transformations and joins efficiently compared to hand-written Spark RDD logic.

---

### **Why Use SparkSQL Over MySQL?**

1. **Scalability**:
   - MySQL is designed for single-node operations, which limits its ability to process large datasets.
   - SparkSQL, running on a distributed Spark cluster, can handle petabytes of data spread across multiple nodes.

2. **Big Data Processing**:
   - SparkSQL works with distributed file systems like HDFS and object storage like S3, allowing it to query massive datasets efficiently.
   - It supports big data-specific formats like Parquet and ORC, which are optimized for analytics.

3. **Integration with Big Data Ecosystems**:
   - SparkSQL integrates easily with other Spark modules (e.g., MLlib for machine learning or GraphX for graph processing) and external data sources.

4. **Speed**:
   - SparkSQL processes data in memory, significantly reducing the time for complex queries compared to MySQL’s disk-based approach.

---

### **Limitations of SparkSQL**

1. **Resource Intensive**:
   - SparkSQL requires a cluster for distributed processing and significant memory, which can be costly.
   - MySQL can be run on simpler, lower-cost infrastructure.

2. **Latency**:
   - While fast for batch and analytics workloads, SparkSQL is not optimized for low-latency transactional queries, where MySQL excels.

3. **Complexity**:
   - Deploying and managing a Spark cluster is more complex than setting up a MySQL instance.

---

### **When to Use SparkSQL?**
- When you need to process and analyze large-scale datasets efficiently.
- For batch processing or ETL tasks over distributed datasets.
- When integrating SQL querying with Spark’s data processing capabilities.

---

### **When to Use MySQL?**
- For small to medium-sized datasets.
- For transactional workloads (e.g., CRUD operations).
- When simplicity and low resource usage are critical.

SparkSQL shines in big data analytics scenarios, while MySQL is better suited for traditional, lightweight, transactional databases.

### **Benefits of Using Spark DataFrame API**
1. **Flexibility:**
   - Combines DataFrame operations with Python logic for advanced transformations.
2. **Performance:**
   - DataFrame operations are optimized through Spark's Catalyst optimizer.
3. **Rich Functions:**
   - Provides a wide range of built-in functions for data manipulation without SQL.


In [ ]:
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [ ]:
# Let's start by reading in a file
postcode_df = spark.read.csv(pc_path)
postcode_df.show(5, False)

Hmm, looks like there's no header. That can be fixed by defining a schema! Fill in the relevant information below.

In [ ]:
schema = T.StructType([
    T.StructField('postcode', T.StringType(), True),
    T.StructField('status', ..., True),
    T.StructField('usertype', ..., True),
    T.StructField('easting', ..., True),
    T.StructField('northing', T.IntegerType(), True),
    T.StructField('positional_quality_indicator', ..., True),
    T.StructField('country', ..., True),
    T.StructField('latitude', T.DecimalType(6, 4), True),
    T.StructField('longitude', ..., True),
    T.StructField('postcode_no_space', ..., True),
    T.StructField('postcode_fixed_width_seven', ..., True),
    T.StructField('postcode_fixed_width_eight', ..., True),
    T.StructField('postcode_area', ..., True),
    T.StructField('postcode_district', ..., True),
    T.StructField('postcode_sector', ..., True),
    T.StructField('outcode', ..., True),
    T.StructField('incode', ..., True)])

In [ ]:
#@title Answer
schema = T.StructType([
    T.StructField('postcode', T.StringType(), True),
    T.StructField('status', T.StringType(), True),
    T.StructField('usertype', T.StringType(), True),
    T.StructField('easting', T.IntegerType(), True),
    T.StructField('northing', T.IntegerType(), True),
    T.StructField('positional_quality_indicator', T.IntegerType(), True),
    T.StructField('country', T.StringType(), True),
    T.StructField('latitude', T.DecimalType(6, 4), True),
    T.StructField('longitude', T.DecimalType(6, 4), True),
    T.StructField('postcode_no_space', T.StringType(), True),
    T.StructField('postcode_fixed_width_seven', T.StringType(), True),
    T.StructField('postcode_fixed_width_eight', T.StringType(), True),
    T.StructField('postcode_area', T.StringType(), True),
    T.StructField('postcode_district', T.StringType(), True),
    T.StructField('postcode_sector', T.StringType(), True),
    T.StructField('outcode', T.StringType(), True),
    T.StructField('incode', T.StringType(), True)])

In [ ]:
postcode_df = spark.read.option("header", "false").csv(pc_path, schema=schema)
postcode_df.show(5, False)

In [ ]:
# count the number of rows
total_rows = postcode_df.count()
print(f"Total Rows: {total_rows:,}")

In [ ]:
postcode_df.select([F.count(F.when(F.isnull(col), col)).alias(col) for col in postcode_df.columns]).show(1, False)

In [ ]:
# count postcodes by area
postcode_df.groupBy("postcode_area")\
    .agg(F.count('postcode').alias("total_postcodes"))\
    .orderBy("total_postcodes", ascending=False)\
    .show(10, False)

In [ ]:
# average lat/long by country
postcode_df\
    .where(F.isnotnull(F.col('latitude')))\
    .where(F.isnotnull(F.col('longitude')))\
    .groupBy("country")\
    .agg(F.round(F.avg('latitude'), 4).alias("avg_lat"),
         F.round(F.avg('longitude')).alias("avg_long"))\
    .orderBy("country", ascending=False)\
    .show(5, False)